In [2]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents


# OLLAMA - using local models with the OpenAI API spec
OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
OPEN_MODEL = "llama3.2"

# OPEN AI - using closed source frontier models with the OpenAI API
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
# OPENAI_BASE_URL = "https://api.openai.com/v1" 
openai = OpenAI()
# openai = OpenAI(base_url=OPENAI_BASE_URL, api_key=api_key)
CLOSED_MODEL = "gpt-4.1-nano"



# 1. fetch summary, fetch links
# 2. llm call to have relevant links only, get strructure links in format link[type, url]
# 3. call beautiful soup to fetch content of each relevant link
# 4. llm call to create brochure in markdown format with all the content

# System Prompt: Instructions set by developers to define AI behavior.
# Purpose: Guide tone, style, and response boundaries.
# Example: "Be a helpful assistant."
# User Prompt: Input from the user for AI interaction.
# Purpose: Request information or actions.
# Example: "What is Python?"
################################   PROMPTS   ##################################################################
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information. If the user is an individual, create a personal brochure about them. Try to find all the work done by the individual and include it in the brochure and how market can benefit from them.
"""
##########################################################################################################

def get_links_user_prompt(url):
    user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

# user prompt:
# print(get_links_user_prompt("https://edwarddonner.com"))

def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {CLOSED_MODEL}")
    response = openai.chat.completions.create(
        model=CLOSED_MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result) #convert to dictonary
    print(f"Found {len(links['links'])} relevant links")
    print(links)
    return links

# print(select_relevant_links("https://edwarddonner.com"))


def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url) #llm call to select relevant links

    # GET CONTENT HEADER = link['type']
    # GET content body for each link['url']
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


# print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt



def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=CLOSED_MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

stream_brochure("Nainesh Goel", "https://naines.github.io")

# def create_brochure(company_name, url):
#     response = openai.chat.completions.create(
#         model="gpt-4.1-mini",
#         messages=[
#             {"role": "system", "content": brochure_system_prompt},
#             {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
#         ],
#     )
#     result = response.choices[0].message.content
#     display(Markdown(result))

# create_brochure("Nainesh Goel", "https://naines.github.io")

# create_brochure("Edward Donner", "https://edwarddonner.com")
#gives 403 after some requests
# create_brochure("HuggingFace", "https://huggingface.co") 

Selecting relevant links for https://naines.github.io by calling gpt-4.1-nano
Found 2 relevant links
{'links': [{'type': 'github profile', 'url': 'https://github.com/Naines'}, {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/naineshgoel/'}]}


# Nainesh Goel - Professional Software Engineer & Technology Enthusiast

## Overview
Nainesh Goel is a dedicated senior software engineer based in Bangalore, working with Fastenal. With extensive expertise in modern software development technologies, Nainesh is passionate about creating innovative solutions and leveraging cutting-edge tools to enhance business operations and software products.

## Technical Expertise
### Primary Tech Stack
- **Java & Spring Framework:** Building robust, scalable backend systems.
- **JavaScript & TypeScript:** Developing dynamic and efficient client-side applications.
- **React.js:** Crafting interactive user interfaces.
- **Docker & Kubernetes:** Containerization and orchestration for scalable deployment.
- **Azure & Azure DevOps:** Cloud solutions and continuous integration/continuous deployment.
- **Struts:** Legacy Java web application framework.

### Secondary Tech Stack
- **Node.js & Golang:** Developing performant server-side applications.
- **Kubernetes:** Orchestrating containerized applications for reliability and scalability.

## Contact & Professional Presence
- **Primary Email:** naineshgoel96@gmail.com
- **Work Email:** ngoel.main@gmail.com
- [GitHub Profile](https://github.com/Naines): Showcase of projects, code, and open-source contributions.
- [LinkedIn Profile](#): Network and professional connection.

## Company & Culture
While specifics about Nainesh's employer, Fastenal, are not detailed, Nainesh emphasizes a focus on technology-driven solutions, continuous learning, and innovation. His work environment likely promotes growth through the use of advanced tools, collaborative problem-solving, and staying abreast of industry best practices.

## Opportunities & Collaboration
Nainesh welcomes job opportunities and professional collaborations. His skill set is well-suited for organizations seeking a passionate and capable senior developer to transform ideas into real-world applications.

---

## Why Collaborate with Nainesh Goel?
- Expertise in full-stack, cloud, and container technology.
- Committed to ongoing learning and embracing new technologies.
- Strong developer presence through GitHub, promoting open-source contributions and community engagement.
- Ready to contribute to projects requiring scalability, security, and modern development practices.

**Let's innovate together!**  
Contact Nainesh today to explore how his skills can benefit your organization or project.